In [1]:
# this notebook will normalize(scale)/ clean the data
# while keeping key level as feature
import pandas as pd
import json
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "mlData"

# src_path = folder_path / "raw" / "BTCUSD-5m-v5-B.jsonl"
src_path = folder_path / "augmented" / "BTCUSD-5m-v6-augmented.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,open,high,low,close,vol,atr42,ts_5m,label,train_mask,...,close_ret1,atr42_pct,ts_15m,ts_45m,15STR_confirmed,45STR_confirmed,barsSince45STR,hDTK_45STR,lDTK_45STR,cDTK_45STR
0,1704037500000,42497.851562,42518.148438,42480.691406,42482.250000,56.512959,NaN,1704037500000,0,0,...,NaN,NaN,1704037500000,1704037500000,0,0,0,0.000882,0.000000,0.000037
1,1704037800000,42482.238281,42500.000000,42411.101562,42414.421875,47.354488,NaN,1704037800000,0,0,...,-0.001597,NaN,1704037500000,1704037500000,0,0,1,0.000455,-0.001638,-0.001560
2,1704038100000,42414.429688,42471.988281,42414.421875,42457.171875,43.228668,NaN,1704038100000,0,0,...,0.001008,NaN,1704037500000,1704037500000,0,0,2,-0.000205,-0.001560,-0.000554
3,1704038400000,42457.171875,42484.828125,42436.468750,42449.601562,74.418266,NaN,1704038400000,0,0,...,-0.000178,NaN,1704038400000,1704037500000,0,0,3,0.000097,-0.001041,-0.000732
4,1704038700000,42449.589844,42488.000000,42449.589844,42487.988281,38.818489,NaN,1704038700000,0,0,...,0.000904,NaN,1704038400000,1704037500000,0,0,4,0.000172,-0.000732,0.000172


In [2]:
# tactical drop
# drop ohl keep c
df['hl_pct'] = (df['high'] - df['low']) / df['close']  # intra-bar range, normalized
df.drop(columns=['open', 'high', 'low'], inplace=True)

In [3]:
print(df.shape)
print(df.columns)
df.head()

(228772, 20)
Index(['timestamp', 'close', 'vol', 'atr42', 'ts_5m', 'label', 'train_mask',
       'body_pct', 'vol_norm', 'close_ret1', 'atr42_pct', 'ts_15m', 'ts_45m',
       '15STR_confirmed', '45STR_confirmed', 'barsSince45STR', 'hDTK_45STR',
       'lDTK_45STR', 'cDTK_45STR', 'hl_pct'],
      dtype='object')


,timestamp,close,vol,atr42,ts_5m,label,train_mask,body_pct,vol_norm,close_ret1,atr42_pct,ts_15m,ts_45m,15STR_confirmed,45STR_confirmed,barsSince45STR,hDTK_45STR,lDTK_45STR,cDTK_45STR,hl_pct
0,1704037500000,42482.250000,56.512959,NaN,1704037500000,0,0,-0.000367,NaN,NaN,NaN,1704037500000,1704037500000,0,0,0,0.000882,0.000000,0.000037,0.000882
1,1704037800000,42414.421875,47.354488,NaN,1704037800000,0,0,-0.001596,NaN,-0.001597,NaN,1704037500000,1704037500000,0,0,1,0.000455,-0.001638,-0.001560,0.002096
2,1704038100000,42457.171875,43.228668,NaN,1704038100000,0,0,0.001008,NaN,0.001008,NaN,1704037500000,1704037500000,0,0,2,-0.000205,-0.001560,-0.000554,0.001356
3,1704038400000,42449.601562,74.418266,NaN,1704038400000,0,0,-0.000178,NaN,-0.000178,NaN,1704038400000,1704037500000,0,0,3,0.000097,-0.001041,-0.000732,0.001139
4,1704038700000,42487.988281,38.818489,NaN,1704038700000,0,0,0.000905,NaN,0.000904,NaN,1704038400000,1704037500000,0,0,4,0.000172,-0.000732,0.000172,0.000904


In [ ]:
# clean
# removing warmup nan
# check label imbalance
import numpy as np
# clean — drop warmup NaN rows (longest window: vol_norm rolling 96 → 95 rows)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
scale_cols = ['atr42', 'vol_norm', 'close_ret1', 'atr42_pct']
df_clean = df.dropna(subset=scale_cols).reset_index(drop=True)


dropped = len(df) - len(df_clean)
print(f"Rows before : {len(df):,}")
print(f"Rows dropped: {dropped:,}  (warmup NaN from rolling windows)")
print(f"Rows after  : {len(df_clean):,}")

# check label imbalance — only on trainable rows (train_mask=1)
trainable = df_clean[df_clean['train_mask'] == 1]
n_up   = (trainable['label'] ==  1).sum()
n_down = (trainable['label'] == -1).sum()
total  = len(trainable)

print(f"\nLabel balance (trainable only):")
print(f"  Up   ( 1) : {n_up:,}  ({n_up   / total * 100:.1f}%)")
print(f"  Down (-1) : {n_down:,}  ({n_down / total * 100:.1f}%)")
print(f"  Total     : {total:,}  ({total / len(df_clean) * 100:.1f}% of clean rows)")
print(f"  Timeout   : {(df_clean['train_mask'] == 0).sum():,}")


Rows before : 228,772
Rows dropped: 95  (warmup NaN from rolling windows)
Rows after  : 228,677

Label balance (trainable only):
  Up   ( 1) : 76,205  (49.5%)
  Down (-1) : 77,788  (50.5%)
  Total     : 153,993  (67.3% of clean rows)
  Timeout   : 74,684


In [5]:
# Nan and Infinity check
assert df_clean.isnull().sum().sum() == 0, "NaN leaked through!"
assert not np.isinf(df_clean.select_dtypes(include='number').values).any(), "Inf leaked through!"

In [6]:
print(df_clean.shape)
# df_clean.tail()
df_clean.columns

(228677, 20)


Index(['timestamp', 'close', 'vol', 'atr42', 'ts_5m', 'label', 'train_mask',
       'body_pct', 'vol_norm', 'close_ret1', 'atr42_pct', 'ts_15m', 'ts_45m',
       '15STR_confirmed', '45STR_confirmed', 'barsSince45STR', 'hDTK_45STR',
       'lDTK_45STR', 'cDTK_45STR', 'hl_pct'],
      dtype='object')

# Export cleaned data

In [7]:
# save the current data in ./data/mlData/clean-v3
# save to JSONL for training
out_path = folder_path / "clean" / "BTCUSD-5m-v6-input.jsonl"
df_clean.to_json(out_path, orient="records", lines=True)
print(f"final shape : {df_clean.shape}")
print(f"Saved {len(df_clean)} rows to {out_path}")


final shape : (228677, 20)
Saved 228677 rows to /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-CandleScience/data/mlData/clean/BTCUSD-5m-v6-input.jsonl
